### Dependencies

In [1]:
!pip install -q rouge-score nltk bert-score requests beautifulsoup4 PyMuPDF

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 96.6 MB/s eta 0:00:00


### Data Collecting (Web Scraping)

#### LOINC

In [2]:
SECTION7_LOINC = "34073-7"
SECTION12_LOINC = "34090-1"

#### Text Retrieval Functions

In [19]:
import re
import requests
from bs4 import BeautifulSoup
from xml.etree import ElementTree
import fitz

def extract_setId(url):
  regMatch = re.search(r'[?&]setid=([a-f0-9]{8}-(?:[a-f0-9]{4}-){3}[a-f0-9]{12})', url)

  if regMatch:
    setId = regMatch.group(1)
    print(f"Extracted ID: {setId}")
    return setId
  return None

def getDailyMedDrugInfo(setId):
  try:
    # API endpoint
    endpoint = f"https://dailymed.nlm.nih.gov/dailymed/services/v2/spls/{setId}.xml"

    response = requests.get(endpoint)
    response.raise_for_status()

    tree = ElementTree.fromstring(response.content)

    section7 = tree.find(".//{urn:hl7-org:v3}code[@code='" f"{SECTION7_LOINC}']/..")
    section12 = tree.find(".//{urn:hl7-org:v3}code[@code='" f"{SECTION12_LOINC}']/..")

    section7Text = None
    section12Text = None

    if section7 is not None:
      section7Text = "".join(section7.itertext()).strip()
    if section12 is not None:
      section12Text = "".join(section12.itertext()).strip()

    return section7Text, section12Text
  except Exception as e:
    return getPdfDrugInfo("https://dailymed.nlm.nih.gov/dailymed/downloadpdffile.cfm?setId="+setId)

def getHtmlDrugInfo(url):
  r = requests.get(url.strip())
  r.raise_for_status()

  soup = BeautifulSoup(r.text, 'html.parser')

  section7 = soup.find('div', attrs={"data-sectionCode": SECTION7_LOINC})
  section12 = soup.find('div', attrs={"data-sectionCode": SECTION12_LOINC})

  section7Text = None
  section12Text = None

  if section7 is not None:
    section7Text = section7.text
  if section12 is not None:
    section12Text = section12.text

  return section7Text, section12Text

def getPdfDrugInfo(url):
    response = requests.get(url)
    found_sections = {"7": "", "12": ""}

    with fitz.open(stream=response.content, filetype="pdf") as doc:
        full_text = ""
        for page in doc:
            full_text += page.get_text("text")

    patterns = {
        "7": r"(?:7\s+DRUG\s+INTERACTIONS)(.*?)(?=\n\d+\s+[A-Z]|\Z)",
        "12": r"(?:12\s+CLINICAL\s+PHARMACOLOGY)(.*?)(?=\n\d+\s+[A-Z]|\Z)"
    }

    for code, pattern in patterns.items():
        match = re.search(pattern, full_text, re.IGNORECASE | re.DOTALL)
        if match:
            found_sections[code] = match.group(1).strip()
        else:
            found_sections[code] = "Section not found"

    return found_sections["7"], found_sections["12"]

#### Load URLs from Excel

In [5]:
import pandas as pd
import io
from google.colab import files

print("Click the button below to upload your Excel file...")
uploaded = files.upload()

df = pd.read_excel(io.BytesIO(list(uploaded.values())[0]))
urls = df['URL'].tolist()
print(f"\nLoaded {len(urls)} URLs successfully\n")

Click the button below to upload your Excel file...


Saving 100 Drug URLs 2026_04_07.xlsx to 100 Drug URLs 2026_04_07 (1).xlsx

Loaded 100 URLs successfully



In [6]:
import requests

def getDrugText(url):
    try:
        response_head = requests.head(url, allow_redirects=True, timeout=10)
        content_type = response_head.headers.get('Content-Type', '').lower()

        if 'application/pdf' in content_type:
            print(f"Detected PDF")
            return getPdfDrugInfo(url)
        elif 'text/html' in content_type:
            print(f"Detected HTML")
            return getHtmlDrugInfo(url)
        else:
            print(f"[!] Unknown content type: {content_type}")

    except Exception as e:
        print(f"[!] Error accessing URL: {e}")

In [21]:
import time
import io

master_drug_data = []
failures = []

def extract_drug_text(url):
  try:
    setId = extract_setId(url)
    if setId is not None:
      section7Text, section12Text = getDailyMedDrugInfo(setId)
    else:
      section7Text, section12Text = getDrugText(url)

    if section7Text is None and section12Text is None:
      return "NA", "NA", False

    return section7Text, section12Text, True
  except Exception as e:
    return "NA", str(e), False

print("Scraping URLs...\n")

for i, url in enumerate(urls):
  print(f"Processing {url}")
  sec7, sec12, success = extract_drug_text(url)

  if success:
    master_drug_data.append({"index": i+1, "url": url, "section7": sec7, "section12": sec12})
    print(f"  [{i+1}/{len(urls)}] OK — {url[:70]}")
  else:
    failures.append({"index": i+1, "url": url, "error": sec12})
    print(f"  [{i+1}/{len(urls)}] FAILED — {url[:70]}, failure: {len(failures)}, error: {sec12}")
  print("----------------------------------------")
  time.sleep(0.15)

print(f"\n✓ Done — {len(master_drug_data)} succeeded, {len(failures)} failed.")

Scraping URLs...

Processing https://labeling.pfizer.com/ShowLabeling.aspx?id=16474 
Detected PDF: https://labeling.pfizer.com/ShowLabeling.aspx?id=16474 
  [1/100] OK — https://labeling.pfizer.com/ShowLabeling.aspx?id=16474 
----------------------------------------
Processing https://dailymed.nlm.nih.gov/dailymed/fda/fdaDrugXsl.cfm?setid=ca73b519-015a-436d-aa3c-af53492825a1&type=display
Extracted ID: ca73b519-015a-436d-aa3c-af53492825a1
  [2/100] OK — https://dailymed.nlm.nih.gov/dailymed/fda/fdaDrugXsl.cfm?setid=ca73b51
----------------------------------------
Processing https://uspl.lilly.com/verzenio/verzenio.html#pi
Detected HTML: https://uspl.lilly.com/verzenio/verzenio.html#pi
  [3/100] FAILED — https://uspl.lilly.com/verzenio/verzenio.html#pi, failure: 1, error: NA
----------------------------------------
Processing https://www.janssenlabels.com/package-insert/product-monograph/prescribing-information/ZYTIGA-pi.pdf
Detected PDF: https://www.janssenlabels.com/package-insert/prod

#### Save Collected Data to CSV

In [25]:
import csv

keys = master_drug_data[0].keys()

with open('collected_drug_data.csv', 'w', newline='') as output_file:
    dict_writer = csv.DictWriter(output_file, keys)
    dict_writer.writeheader()
    dict_writer.writerows(master_drug_data)

print("Data saved to 'collected_drug_data.csv'")

Data saved to 'collected_drug_data.csv'


In [26]:
import csv

keys = failures[0].keys()

with open('failures_drug_data.csv', 'w', newline='') as output_file:
    dict_writer = csv.DictWriter(output_file, keys)
    dict_writer.writeheader()
    dict_writer.writerows(failures)

print("Failures saved to 'failures_drug_data.csv'")

Failures saved to 'failures_drug_data.csv'


#### Optionally Provide a CSV Drug Data File

In [2]:
from google.colab import files
import csv

print("Click the button below to upload the CSV file...")
uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))

filename = next(iter(uploaded))

master_drug_data = []
with open(filename, newline='') as csvfile:
    reader = csv.DictReader(csvfile)
    for row in reader:
        master_drug_data.append(row)

print(master_drug_data)

Click the button below to upload the CSV file...


Saving collected_drug_data.csv to collected_drug_data.csv
User uploaded file "collected_drug_data.csv" with length 553968 bytes
[{'index': '1', 'url': 'https://labeling.pfizer.com/ShowLabeling.aspx?id=16474', 'section7': '7.1 \nPotential for PAXLOVID to Affect Other Drugs \n \n7.2 \nPotential for Other Drugs to Affect PAXLOVID \n \n7.3 \nEstablished and Other Potentially Significant Drug \nInteractions', 'section12': '12.1 Mechanism of Action \n12.2 Pharmacodynamics \n \n12.3 Pharmacokinetics  \n \n12.4 Microbiology'}, {'index': '2', 'url': 'https://dailymed.nlm.nih.gov/dailymed/fda/fdaDrugXsl.cfm?setid=ca73b519-015a-436d-aa3c-af53492825a1&type=display', 'section7': '7 DRUG INTERACTIONS\n               \n               \n                  \n                     \n                        \n                           \n                              •Methadone: An increased methadone dose may be required in a small number of patients. (7.1)\n                           \n                  

### LongT5 Inference

#### Remove MedGemma From Memory

In [3]:
import gc
import torch

try:
    del model_mg
    del pipe_mg
    gc.collect()
    torch.cuda.empty_cache()
    print(f"GPU cleared. Free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")
except:
    print("No MedGemma in memory, continuing...")

No MedGemma in memory, continuing...


#### Iterate Over Drug Data

In [5]:
import torch
import time
import warnings
import textwrap
from transformers import AutoTokenizer, LongT5ForConditionalGeneration

print("\nLoading Long-T5...")
tokenizer_lt5 = AutoTokenizer.from_pretrained("google/long-t5-tglobal-base")
model_lt5 = LongT5ForConditionalGeneration.from_pretrained(
    "google/long-t5-tglobal-base",
    torch_dtype=torch.float16
).to("cuda")
print("Long-T5 loaded!\n")

def summarize_long_t5(text, max_words=150):
    prompt = (
        "Generate a detailed abstractive summary of the following medical text. "
        "Do not copy sentences directly. Rewrite the key information in your own words, "
        "covering indications, dosing, warnings, side effects, and drug interactions: "
        + text
    )
    inputs = tokenizer_lt5(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=16384
    ).to(model_lt5.device)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        output_ids = model_lt5.generate(
            inputs["input_ids"],
            min_new_tokens=80,
            max_new_tokens=250,
            num_beams=4,
            length_penalty=2.0,
            no_repeat_ngram_size=4,
            early_stopping=False
        )
    return tokenizer_lt5.decode(output_ids[0], skip_special_tokens=True)

print("Running Long-T5 on all drugs (this will take a while, do not interrupt)...")
lt5_results = []
lt5_times   = []

num_texts = len(master_drug_data) * 2

for i, data in enumerate(master_drug_data):
    for j, text in enumerate([data["section7"], data["section12"]]):
      start = time.time()
      output = summarize_long_t5(text)
      elapsed = round(time.time() - start, 2)
      lt5_results.append(output)
      lt5_times.append(elapsed)
      master_drug_data[i][f"LongT5_Summary_{j}"] = output
      master_drug_data[i][f"LongT5_Time_{j}"] = elapsed

      if (i + 1 + j) % 5 == 0 or (i + 1 + j) == num_texts:
          print(f"  Progress: {i+1+j}/{num_texts} done...")

lt5_avg_time = round(sum(lt5_times)/len(lt5_times),2)

print(f"\n✓ LongT5 done. Avg time: {lt5_avg_time}s")


Loading Long-T5...


Loading weights:   0%|          | 0/297 [00:00<?, ?it/s]

Long-T5 loaded!

Running Long-T5 on all drugs (this will take a while, do not interrupt)...
  Progress: 10/144 done...
  Progress: 10/144 done...
  Progress: 20/144 done...
  Progress: 20/144 done...
  Progress: 30/144 done...
  Progress: 30/144 done...
  Progress: 40/144 done...
  Progress: 40/144 done...
  Progress: 50/144 done...
  Progress: 50/144 done...
  Progress: 60/144 done...
  Progress: 60/144 done...
  Progress: 70/144 done...
  Progress: 70/144 done...

✓ LongT5 done. Avg time: 6.24s


#### Save Results to CSV

In [6]:
import csv

keys = master_drug_data[0].keys()

with open('drug_data_LongT5_results.csv', 'w', newline='') as output_file:
    dict_writer = csv.DictWriter(output_file, keys)
    dict_writer.writeheader()
    dict_writer.writerows(master_drug_data)

print("Data saved to 'drug_data_LongT5_results.csv'")

Data saved to 'drug_data_LongT5_results.csv'


### MedGemma Inference

#### Remove LongT5 From Memory

In [7]:
import os
import gc
import torch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

try:
    del model_lt5
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print(f"GPU cleared. Free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

GPU cleared. Free: 41.84 GB


#### Load MedGemma

In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

print("\nLoading MedGemma...")
tokenizer_mg = AutoTokenizer.from_pretrained("google/medgemma-4b-it")
model_mg = AutoModelForCausalLM.from_pretrained(
    "google/medgemma-4b-it",
    dtype=torch.bfloat16,
    device_map="auto"
)
model_mg.generation_config.max_length = None
pipe_mg = pipeline(
    "text-generation",
    model=model_mg,
    tokenizer=tokenizer_mg,
    max_new_tokens=150
)
print("MedGemma loaded!\n")


Loading MedGemma...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


MedGemma loaded!



#### Iterate Over Drug Data

In [9]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import gc
import torch

def chunk_text(text, chunk_size=150):
    words = text.split()
    return [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

def medgemma_summarize_chunk(chunk):
    messages = [{"role": "user", "content": (
        "Summarize this medical text in 2-3 sentences capturing key clinical info:\n\n"
        + chunk
    )}]
    formatted = tokenizer_mg.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        output = pipe_mg(formatted, return_full_text=False)
    return output[0]["generated_text"].strip()

def medgemma_base_prompt(text, max_words=150):
    chunks = chunk_text(text, chunk_size=150)
    if len(chunks) == 1:
        return medgemma_summarize_chunk(chunks[0])
    summaries = [medgemma_summarize_chunk(c) for c in chunks]
    combined  = " ".join(summaries)
    messages  = [{"role": "user", "content": (
        f"Combine these into one medical summary paragraph of {max_words} words "
        f"or less. No repeated information:\n\n" + combined
    )}]
    formatted = tokenizer_mg.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        output = pipe_mg(formatted, return_full_text=False)
    return output[0]["generated_text"].strip()

def medgemma_summarize_text(text):
    prompt = (
        "Generate a detailed abstractive summary of the following medical text. "
        "Do not copy sentences directly. Rewrite the key information in your own words, "
        "covering indications, dosing, warnings, side effects, and drug interactions: "
        + text
    )
    messages  = [{"role": "user", "content": prompt}]
    formatted = tokenizer_mg.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        output = pipe_mg(formatted, return_full_text=False)
    return output[0]["generated_text"].strip()

print("Running MedGemma on all drugs (do not interrupt)...")
mg_results = []
mg_times   = []

num_texts = len(master_drug_data) * 2

for i, data in enumerate(master_drug_data):
    if i > 0 and i % 5 == 0:
        gc.collect()
        torch.cuda.empty_cache()

    for j, text in enumerate([data["section7"], data["section12"]]):
        start = time.time()
        output = medgemma_summarize_text(text)
        elapsed = round(time.time() - start, 2)
        mg_results.append(output)
        mg_times.append(elapsed)
        master_drug_data[i][f"MedGemma_Summary_{j}"] = output
        master_drug_data[i][f"MedGemma_Time_{j}"] = elapsed

        if (i + 1 + j) % 5 == 0 or (i + 1 + j) == num_texts:
            free_gb = torch.cuda.mem_get_info()[0]/1e9
            print(f"  Progress: {i+1+j}/{num_texts} done — GPU free: {free_gb:.2f} GB")

mg_avg_time = round(sum(lt5_times)/len(lt5_times),2)

print(f"\n✓ MedGemma done. Avg time: {mg_avg_time}s")

Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running MedGemma on all drugs (do not interrupt)...


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

  Progress: 5/144 done — GPU free: 32.14 GB


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

  Progress: 70/144 done — GPU free: 32.59 GB


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Progress: 70/144 done — GPU free: 32.59 GB


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



✓ MedGemma done. Avg time: 6.24s


#### Save Results to CSV

In [10]:
import csv

keys = master_drug_data[0].keys()

with open('drug_data_MedGemma_results.csv', 'w', newline='') as output_file:
    dict_writer = csv.DictWriter(output_file, keys)
    dict_writer.writeheader()
    dict_writer.writerows(master_drug_data)

print("Data saved to 'drug_data_MedGemma_results.csv'")

Data saved to 'drug_data_MedGemma_results.csv'


### Results

#### Determine BERTScores

In [15]:
import logging
import os
from bert_score import score as bert_score

logging.getLogger("transformers").setLevel(logging.ERROR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def wrap_output(text, width=80, indent="  "):
    return "\n".join(textwrap.wrap(
        text.strip(), width=width,
        initial_indent=indent, subsequent_indent=indent
    ))

def get_bertscore_batch(hypotheses, references):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        _, _, F1 = bert_score(
            hypotheses, references, lang="en", verbose=False
        )
    return [round(f.item(), 4) for f in F1]

references = []
for data in master_drug_data:
    for text in [data["section7"], data["section12"]]:
        references.append(text)

print(len(lt5_results), len(mg_results), len(references))

print("Scoring all outputs with BERTScore...")
lt5_scores = get_bertscore_batch(lt5_results, references)
mg_scores  = get_bertscore_batch(mg_results,  references)
print("Done.\n")


144 144 144
Scoring all outputs with BERTScore...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

AttributeError: RobertaTokenizer has no attribute build_inputs_with_special_tokens

In [16]:
!pip install transformers==5.5.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 86.9 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


#### Select Three Examples

In [ ]:
import random

showcase_indices = []

for i in range(4):
    random_index = random.randint(0, len(master_drug_data) - 1)
    showcase_indices.append(random_index)

showcase_cases = [master_drug_data[i] for i in showcase_indices if i < len(master_drug_data)]

print(f"Showcase cases  : {len(showcase_cases) * 2}")
print(f"Total test cases: {len(master_drug_data) * 2}")
for c in showcase_cases:
    print(f"  [{c['index']}] {c['url'][:70]}")

#### Showcase Examples

In [ ]:
print("="*70)
print("  SHOWCASE — 3 EXAMPLE SUMMARIES")
print("="*70)

for case in showcase_cases:
    idx = next(i for i, c in enumerate(master_drug_data) if c["url"] == case["url"])

    print(f"\n{'─'*70}")
    print(f"  DRUG {idx+1}")
    print(f"  Source: {case['url'][:70]}")
    print(f"{'─'*70}")

    print(f"\n  ORIGINAL TEXT (first 150 words):")
    print(wrap_output(" ".join(case["text"].split()[:150])))

    print(f"\n  LONG-T5 SUMMARY:")
    print(wrap_output(lt5_results[idx]))
    print(f"\n  ⏱ Time: {lt5_times[idx]}s  |  BERTScore F1: {lt5_scores[idx]}")

    print(f"\n  MEDGEMMA SUMMARY:")
    print(wrap_output(mg_results[idx]))
    print(f"\n  ⏱ Time: {mg_times[idx]}s  |  BERTScore F1: {mg_scores[idx]}")


#### Overall Comparison

In [ ]:
avg_lt5_score = round(sum(lt5_scores) / len(lt5_scores), 4)
avg_mg_score  = round(sum(mg_scores)  / len(mg_scores),  4)
avg_lt5_time  = round(sum(lt5_times)  / len(lt5_times),  2)
avg_mg_time   = round(sum(mg_times)   / len(mg_times),   2)
winner        = "Long-T5"  if avg_lt5_score > avg_mg_score else "MedGemma"
faster        = "Long-T5"  if avg_lt5_time  < avg_mg_time  else "MedGemma"

print(f"\n{'='*70}")
print("  OVERALL RESULTS — 97 DRUG PRESCRIBING INFORMATION DOCUMENTS")
print(f"{'='*70}")
print(f"  {'Metric':<30} {'Long-T5':>10} {'MedGemma':>10}")
print(f"  {'─'*50}")
print(f"  {'Avg BERTScore F1':<30} {avg_lt5_score:>10} {avg_mg_score:>10}")
print(f"  {'Avg Time per Summary (s)':<30} {avg_lt5_time:>10} {avg_mg_time:>10}")
print(f"  {'Total Drugs Evaluated':<30} {len(all_cases):>10} {len(all_cases):>10}")
print(f"{'─'*70}")
print(f"  More Accurate Model : {winner}")
print(f"  Faster Model        : {faster}")
print(f"{'='*70}")